In [2]:
import io
import re
import pandas as pd
import requests


NASDAQ_LISTED_URL = "https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqlisted.txt"
OTHER_LISTED_URL  = "https://www.nasdaqtrader.com/dynamic/SymDir/otherlisted.txt"


def _download_text(url: str, timeout: int = 30) -> str:
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    # 文件是文本，直接按原样拿
    return r.text


def _parse_nasdaq_listed(raw: str) -> pd.DataFrame:
    """
    nasdaqlisted.txt 结构：表头行 + 数据行 + 末尾 'File Creation Time: ...'
    分隔符：'|'
    """
    lines = raw.splitlines()
    # 去掉最后一行 "File Creation Time: ..."
    lines = [ln for ln in lines if not ln.startswith("File Creation Time:")]
    df = pd.read_csv(io.StringIO("\n".join(lines)), sep="|", dtype=str)

    # 常见字段：Symbol, Security Name, Market Category, ETF, Test Issue, Financial Status, Round Lot Size, NextShares
    # 清理空白
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    # 过滤测试标的
    if "Test Issue" in df.columns:
        df = df[df["Test Issue"].str.upper().fillna("N") != "Y"]

    # 统一列名
    df = df.rename(columns={
        "Symbol": "ticker",
        "Security Name": "name",
        "ETF": "is_etf",
    })

    df["exchange"] = "NASDAQ"
    return df


def _parse_other_listed(raw: str) -> pd.DataFrame:
    """
    otherlisted.txt：同样是 '|' 分隔，末尾也有 File Creation Time
    字段里有：ACT Symbol, Security Name, Exchange, ETF, Test Issue 等
    """
    lines = raw.splitlines()
    lines = [ln for ln in lines if not ln.startswith("File Creation Time:")]
    df = pd.read_csv(io.StringIO("\n".join(lines)), sep="|", dtype=str)
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    # 过滤测试标的
    if "Test Issue" in df.columns:
        df = df[df["Test Issue"].str.upper().fillna("N") != "Y"]

    # ACT Symbol 是交易代码
    df = df.rename(columns={
        "ACT Symbol": "ticker",
        "Security Name": "name",
        "ETF": "is_etf",
        "Exchange": "exchange",
    })

    return df


def get_us_listed_universe(include_etf: bool = True) -> pd.DataFrame:
    raw_nasdaq = _download_text(NASDAQ_LISTED_URL)
    raw_other  = _download_text(OTHER_LISTED_URL)

    df1 = _parse_nasdaq_listed(raw_nasdaq)
    df2 = _parse_other_listed(raw_other)

    # 合并
    df = pd.concat([df1, df2], ignore_index=True)

    # 规范化：ticker 去掉空白
    df["ticker"] = df["ticker"].astype(str).str.strip()

    # 过滤掉明显的无效 ticker（可按你需要调整）
    df = df[df["ticker"].str.len() > 0]
    df = df[~df["ticker"].str.contains(r"\s", regex=True)]

    # ETF 过滤（可选）
    if not include_etf and "is_etf" in df.columns:
        df = df[df["is_etf"].str.upper().fillna("N") != "Y"]

    # 去重（同一ticker偶尔可能重复，保留第一条）
    df = df.drop_duplicates(subset=["ticker"], keep="first")

    # 只保留你关心的列（你后面可以扩展）
    keep_cols = [c for c in ["ticker", "name", "exchange", "is_etf"] if c in df.columns]
    df = df[keep_cols].sort_values("ticker").reset_index(drop=True)

    return df


if __name__ == "__main__":
    universe = get_us_listed_universe(include_etf=True)
    print(universe.head(20))
    print(f"\nTotal tickers: {len(universe):,}")

    # 导出到本地
    universe.to_csv("us_listed_universe.csv", index=False, encoding="utf-8-sig")
    print("\nSaved: us_listed_universe.csv")


   ticker                                               name exchange is_etf
0       A            Agilent Technologies, Inc. Common Stock        N      N
1      AA                     Alcoa Corporation Common Stock        N      N
2     AAA     Alternative Access First Priority CLO Bond ETF        P      Y
3    AAAA            Amplius Aggressive Asset Allocation ETF        Z      Y
4    AAAC                               Columbia AAA CLO ETF        P      Y
5    AAAU             Goldman Sachs Physical Gold ETF Shares        Z      Y
6    AACB  Artius II Acquisition Inc. - Class A Ordinary ...   NASDAQ      N
7   AACBR                Artius II Acquisition Inc. - Rights   NASDAQ      N
8   AACBU                 Artius II Acquisition Inc. - Units   NASDAQ      N
9    AACG  ATA Creativity Global - American Depositary Sh...   NASDAQ      N
10   AADR                AdvisorShares Dorsey Wright ADR ETF   NASDAQ      Y
11   AAEQ                    Alpha Architect US Equity 2 ETF   NASDAQ      Y

C:\Users\Administrator\AppData\Local\Temp\2\ipykernel_27464\1842573704.py:30: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
C:\Users\Administrator\AppData\Local\Temp\2\ipykernel_27464\1842573704.py:55: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
